In [ ]:
import time

notebook_start_time = time.time()

# Set up environment

In [ ]:
import sys
from pathlib import Path


def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False


def clone_repository() -> None:
    !git clone https://github.com/decodingml/hands-on-recommender-system.git
    %cd hands-on-recommender-system/


def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml


if is_google_colab():
    clone_repository()
    install_dependencies()

    root_dir = str(Path().absolute())
    print("⛳️ Google Colab environment")
else:
    root_dir = str(Path().absolute().parent)
    print("⛳️ Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    print(f"Adding the following directory to the PYTHONPATH: {root_dir}")
    sys.path.append(root_dir)

⛳️ Local environment


# Training pipeline: Training ranking model </span>

In this notebook, you will train a ranking model using gradient boosted trees. 

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

warnings.filterwarnings("ignore")

from loguru import logger

from recsys import hopsworks_integration, training
from recsys.config import settings

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Constants

In [ ]:
from pprint import pprint

pprint(dict(settings))

{'CUSTOMER_DATA_SIZE': <CustomerDatasetSize.SMALL: 'SMALL'>,
 'CUSTOM_HOPSWORKS_INFERENCE_ENV': 'custom_env_name',
 'FEATURES_EMBEDDING_MODEL_ID': 'all-MiniLM-L6-v2',
 'HOPSWORKS_API_KEY': SecretStr('**********'),
 'OPENAI_API_KEY': SecretStr(''),
 'OPENAI_MODEL_ID': 'gpt-4o-mini',
 'RANKING_DATASET_VALIDATON_SPLIT_SIZE': 0.1,
 'RANKING_EARLY_STOPPING_ROUNDS': 5,
 'RANKING_ITERATIONS': 100,
 'RANKING_LEARNING_RATE': 0.2,
 'RANKING_MODEL_TYPE': 'ranking',
 'RANKING_SCALE_POS_WEIGHT': 10,
 'RECSYS_DIR': WindowsPath('c:/Users/PC/OneDrive - National Economics University/Tài liệu/personalized-recommender-course/recsys'),
 'TWO_TOWER_DATASET_TEST_SPLIT_SIZE': 0.1,
 'TWO_TOWER_DATASET_VALIDATON_SPLIT_SIZE': 0.1,
 'TWO_TOWER_LEARNING_RATE': 0.01,
 'TWO_TOWER_MODEL_BATCH_SIZE': 2048,
 'TWO_TOWER_MODEL_EMBEDDING_SIZE': 16,
 'TWO_TOWER_NUM_EPOCHS': 10,
 'TWO_TOWER_WEIGHT_DECAY': 0.001}


## Connect to Hopsworks Feature Store 

In [ ]:
import hopsworks

hopsworks_host = "eu-west.cloud.hopsworks.ai"

project = hopsworks.login(
    host=hopsworks_host,
    api_key_value=settings.HOPSWORKS_API_KEY.get_secret_value()
)

# Thêm logic này để tự tạo project nếu tài khoản bị trống
if project is None:
    project = hopsworks.create_project("recommender_system", description="H&M Recommender")

fs = project.get_feature_store()

2026-03-10 14:36:28,946 INFO: Closing external client and cleaning up certificates.
2026-03-10 14:36:29,098 INFO: Connection closed.
2026-03-10 14:36:29,106 INFO: Initializing external client
2026-03-10 14:36:29,106 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-03-10 14:36:32,605 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31873


# Getting the training data

In [ ]:
feature_view_ranking = hopsworks_integration.feature_store.create_ranking_feature_views(
    fs
)

In [ ]:
X_train, X_val, y_train, y_val = feature_view_ranking.train_test_split(
    test_size=settings.RANKING_DATASET_VALIDATON_SPLIT_SIZE,
    description="Ranking training dataset",
)
X_train.head(3)

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (6.03s) 
2026-03-10 14:37:06,256 INFO: Computing insert statistics
2026-03-10 14:37:06,624 INFO: Computing insert statistics


,age,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name,month_sin,month_cos
0,48.0,Polo shirt,Garment Upper body,Solid,White,Light,White,Jersey Fancy,Menswear,Menswear,Contemporary Smart,Jersey Fancy,0.5,-8.660254e-01
1,50.0,Dress,Garment Full body,Solid,Dark Beige,Dark,Beige,Knitwear,Ladieswear,Ladieswear,Womens Everyday Collection,Knitwear,-0.5,8.660254e-01
2,26.0,Top,Garment Upper body,Solid,White,Light,White,Jersey,Ladieswear,Ladieswear,Womens Tailoring,Jersey Fancy,-1.0,-1.836970e-16


In [ ]:
y_train.head(3)

,label
0,1
1,1
2,1


# Training the ranking model

Let's train the ranking model:

In [ ]:
model = training.ranking.RankingModelFactory.build()
trainer = training.ranking.RankingModelTrainer(
    model=model, train_dataset=(X_train, y_train), eval_dataset=(X_val, y_val)
)

In [ ]:
trainer.fit()

0:	learn: 0.5148683	test: 0.5149081	best: 0.5149081 (0)	total: 165ms	remaining: 16.3s
1:	learn: 0.3950439	test: 0.3951204	best: 0.3951204 (1)	total: 246ms	remaining: 12s
2:	learn: 0.3092757	test: 0.3093850	best: 0.3093850 (2)	total: 346ms	remaining: 11.2s
3:	learn: 0.2454958	test: 0.2456312	best: 0.2456312 (3)	total: 439ms	remaining: 10.5s
4:	learn: 0.1969518	test: 0.1971111	best: 0.1971111 (4)	total: 501ms	remaining: 9.51s
5:	learn: 0.1593917	test: 0.1595784	best: 0.1595784 (5)	total: 547ms	remaining: 8.57s
6:	learn: 0.1300034	test: 0.1302172	best: 0.1302172 (6)	total: 618ms	remaining: 8.21s
7:	learn: 0.1068122	test: 0.1070480	best: 0.1070480 (7)	total: 699ms	remaining: 8.04s
8:	learn: 0.0884084	test: 0.0886619	best: 0.0886619 (8)	total: 802ms	remaining: 8.11s
9:	learn: 0.0737218	test: 0.0739996	best: 0.0739996 (9)	total: 847ms	remaining: 7.62s
10:	learn: 0.0619787	test: 0.0622789	best: 0.0622789 (10)	total: 904ms	remaining: 7.32s
11:	learn: 0.0525785	test: 0.0529006	best: 0.0529006 (

## Evaluating the ranking model

Next, you'll evaluate how well the model performs on the validation data using metrics for classification such as precision, recall and f1-score:

In [ ]:
metrics = trainer.evaluate(log=True)

2026-03-10 14:37:22.549 | INFO     | recsys.training.ranking:evaluate:62 -               precision    recall  f1-score   support

           0       1.00      1.00      1.00     19883
           1       0.95      1.00      0.98      1946

    accuracy                           1.00     21829
   macro avg       0.98      1.00      0.99     21829
weighted avg       1.00      1.00      1.00     21829



It can be seen that the model has a low F1-score on the positive class (higher is better). The performance could potentially be improved by adding more features to the dataset, e.g. image embeddings.

Let's see which features your model considers important.

In [ ]:
trainer.get_feature_importance()

{'month_cos': 62.71082892878465,
 'month_sin': 35.47164233952842,
 'age': 0.42030031033362303,
 'product_group_name': 0.26463735632256546,
 'index_name': 0.20643825682732267,
 'section_name': 0.1914622939853796,
 'department_name': 0.19129755418303543,
 'perceived_colour_master_name': 0.16251189772141175,
 'graphical_appearance_name': 0.12016199083586425,
 'garment_group_name': 0.08767264772542131,
 'perceived_colour_value_name': 0.08664028306164895,
 'index_group_name': 0.04084936086397908,
 'colour_group_name': 0.028482852255127766,
 'product_type_name': 0.017073927571579075}

## Uploading the model to Hopsworks model registry 

In [ ]:
mr = project.get_model_registry()

In [ ]:
ranking_module = hopsworks_integration.ranking_serving.HopsworksRankingModel(
    model=model
)
ranking_module.register(mr, feature_view_ranking, X_train, metrics)

Uploading c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course\notebooks\ranking_model.pkl: 100.000%|██████████| 413412/413412 elapsed<00:02 remaining<00:00
Uploading c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course\notebooks\input_example.json: 100.000%|██████████| 433/433 elapsed<00:01 remaining<00:00
Uploading c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course\notebooks\model_schema.json: 100.000%|██████████| 1338/1338 elapsed<00:01 remaining<00:00
Model export complete: 100%|██████████| 6/6 [00:16<00:00,  2.72s/it]                   

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/31873/models/ranking_model/2


## Inspecting the model in the Hopsworks model registry </span>

View results in [Hopsworks Serverless](https://rebrand.ly/serverless-github): **Data Science → Model Registry**

---

In [ ]:
notebook_end_time = time.time()
notebook_execution_time = notebook_end_time - notebook_start_time

logger.info(
    f"⌛️ Notebook Execution time: {notebook_execution_time:.2f} seconds ~ {notebook_execution_time / 60:.2f} minutes"
)

2026-03-10 14:37:45.024 | INFO     | __main__:<module>:4 - ⌛️ Notebook Execution time: 80.23 seconds ~ 1.34 minutes
